# Lecture d'un historique MARTHE

## lecture des données

In [ ]:
import pandas as pd

In [ ]:
df_obs = pd.read_csv(
    'data/00471X0095.csv',
    sep=',',
    encoding='utf-8',
    parse_dates=True,
    index_col=0
) #id(df) pour vérifier si même objet en memoire

In [ ]:
df_sim = pd.read_csv(
    'data/historiq.prn',
    encoding='ISO-8859-1',
    sep=r'\t',
    skiprows=[0],
    header=list(range(4)),
    parse_dates=True,
    index_col=0,
    dayfirst=True
)

df_sim.columns.names = ['variable', 'xy', 'grid', 'name']
df_sim.index.name = 'date'
df_sim

## extraction de données

In [ ]:
h_sim = df_sim.xs(level='variable', key=('Charge'), axis='columns')#.squeeze()
h_sim
#h_sim.name = 'Simulation'

In [ ]:
h_sim1 = df_sim.loc[:, df_sim.columns.get_level_values('variable').isin(['Charge'])].squeeze()#, , 'Débit_Rivi'
#h_sim
h_sim = df_sim.loc[:, ('Charge', slice(None), slice(None), '00471X0010/H1')] # slice(None, None, None) départ, fin, pas
h_sim.columns = ['Simulation']
h_sim

In [ ]:
h_obs = df_obs['value'][df_sim.index.min():df_sim.index.max()]
h_obs.name = 'Observation'
h_obs = h_obs.resample('D').mean()

## Plot

In [ ]:
from matplotlib import pyplot as plt
fig = plt.figure()
ax = fig.add_subplot(111)
h_obs.plot(ax=ax, marker='.', markersize=2, color='k')
h_sim.plot(ax=ax, color='r')
ax.set_ylabel('hydraulic head [m]')
ax.grid()
ax.legend()

## calcul de scores

In [ ]:
df = pd.concat([h_obs, h_sim], axis=1)
print(df)

In [ ]:
df_skill = df.dropna(how='any', axis=0)

In [ ]:
df_skill

In [ ]:
df_skill.corr()

In [ ]:
means = df_skill.mean(axis=0)
(means['Simulation'] - means['Observation']).round(2)

In [ ]:
import numpy as np
rmse = np.sqrt(((df_skill['Simulation'] - df_skill['Observation'])**2).mean())
rmse

In [ ]:
import numpy as np
obs = df_skill['Observation']
sim = df_skill['Simulation']
nash = 1 - np.sum((sim - obs)**2) /np.sum((obs - obs.mean())**2)
nash.round(2)

In [ ]:
# q_sim = df_sim.xs(level='variable', key='Débit_Rivi', axis='columns').squeeze()
q_sim = df_sim.loc[:, ('Débit_Rivi', slice(None), slice(None), 'E6470092')]
q_sim.name = 'Simulation'
q_sim.plot(grid=True, title='Débit simulé', ylabel='m3/s')

In [ ]:
q_sim.groupby(h_sim.index.month).mean().plot(grid=True, title='Cycle saisonnier mensuel', style='-o', ylabel='m3/s')

## introduction à martpy

Martpy est un package python développé par le BRGM pour faciliter l'utilisation de Marthe dans un environnement python.
Cette librairie fait suite à de nombreux développements (librairies internes brgm, projet pymarthe, etc.) et
est actuellement en cours de développement (beta - 03/2026).

`martpy` permet de lire les fichiers d'entrées et de sorties de MARTHE.

In [1]:
import martpy as mart

In [2]:
mart.read_prn('data/historiq.prn')

variable         Charge                                            \
coords          X56_Y20       X75_Y15        X51_Y52      X36_Y56   
grid          Main_Grid     Main_Grid      Main_Grid    Main_Grid   
name       00238X0037/F 00245X0002/S1 00323X0078/PZ1 00326X0018/P   
date                                                                
1991-07-31     6.369383      20.81524       6.588973     22.94043   
1991-08-01     6.364040      20.79644       6.584376     22.92215   
1991-08-02     6.364040      20.79644       6.584376     22.92215   
1991-08-03     6.364040      20.79644       6.584376     22.92215   
1991-08-04     6.364040      20.79644       6.584376     22.92215   
...                 ...           ...            ...          ...   
2024-07-27     6.875373      23.86695       7.470657     30.92803   
2024-07-28     6.875373      23.86695       7.470657     30.92803   
2024-07-29     6.875373      23.86695       7.470657     30.92803   
2024-07-30     6.875373      23.86695       7.470657     30.92803   
2024-07-31     6.875373      23.86695       7.470657     30.92803   

variable                                                               \
coords           X50_Y63          X71_Y41       X90_Y35       X89_Y42   
grid           Main_Grid        Main_Grid     Main_Grid     Main_Grid   
name       00327X0062/P1 00331X0051/F.SEP 00332X0006/S1 00332X0007/S1   
date                                                                    
1991-07-31      33.72694         24.87332      43.68255      50.97055   
1991-08-01      33.70992         24.85227      43.66664      50.96547   
1991-08-02      33.70992         24.85227      43.66664      50.96547   
1991-08-03      33.70992         24.85227      43.66664      50.96547   
1991-08-04      33.70992         24.85227      43.66664      50.96547   
...                  ...              ...           ...           ...   
2024-07-27      39.33656         29.03853      48.21200      54.00247   
2024-07-28      39.33656         29.03853      48.21200      54.00247   
2024-07-29      39.33656         29.03853      48.21200      54.00247   
2024-07-30      39.33656         29.03853      48.21200      54.00247   
2024-07-31      39.33656         29.03853      48.21200      54.00247   

variable                               ... Débit_Rivi                          \
coords           X82_Y50               ...   X159_Y97    X436_Y41    X485_Y40   
grid           Main_Grid               ...  Main_Grid Gigogne : 2 Gigogne : 2   
name       00332X0009/S1 00332X0036/P  ...   E6397030    E6351402    E6330950   
date                                   ...                                      
1991-07-31      22.49310     21.93686  ...   0.587971    4.935755    0.982332   
1991-08-01      22.48547     21.92943  ...   0.583453    4.910962    0.976762   
1991-08-02      22.48547     21.92943  ...   0.583453    4.910962    0.976762   
1991-08-03      22.48547     21.92943  ...   0.583453    4.910962    0.976762   
1991-08-04      22.48547     21.92943  ...   0.583453    4.910962    0.976762   
...                  ...          ...  ...        ...         ...         ...   
2024-07-27      24.38513     23.72032  ...   1.245103   14.278350    2.349900   
2024-07-28      24.38513     23.72032  ...   1.245103   14.278350    2.349900   
2024-07-29      24.38513     23.72032  ...   1.245103   14.278350    2.349900   
2024-07-30      24.38513     23.72032  ...   1.245103   14.278350    2.349900   
2024-07-31      24.38513     23.72032  ...   1.245103   14.278350    2.349900   

variable                                                   Flux_ETR  \
coords       X151_Y232   X148_Y234   X57_Y22    X207_Y151  X156_Y80   
grid       Gigogne : 3 Gigogne : 3 Main_Grid    Main_Grid Main_Grid   
name          E6351420      Hambis  E6498315 Avre_St_Mart         @   
date                                                                  
1991-07-31    0.416919    1.162463  0.455882     0.129443       0.0   
199

In [3]:
mart.read_prn('data/histobil_debit.prn')

Entr_Limit  Sort_Limit  Entr_Inject_fix  Sort_Pompag_fix  \
zone   date                                                                   
global 1995-07-31    192385.4   -10941.12              0.0           0.0000   
       1995-08-01    192385.4   -10941.12              0.0        -625.1014   
       1995-08-02    192385.5   -10941.12              0.0        -625.1014   
       1995-08-03    192385.6   -10941.12              0.0        -625.1014   
       1995-08-04    192385.9   -10941.11              0.0        -625.1014   
...                       ...         ...              ...              ...   
       2012-07-27    174114.3   -16326.97              0.0        -520.8978   
       2012-07-28    174163.7   -16302.75              0.0        -520.8978   
       2012-07-29    174217.0   -16277.21              0.0        -520.8978   
       2012-07-30    174273.8   -16250.49              0.0        -520.8978   
       2012-07-31    174333.7   -16222.73              0.0        -520.8978   

                   Réduct_Pompage  Désaturation  Débord/Suintem  \
zone   date                                                       
global 1995-07-31             0.0           0.0       -79599.67   
       1995-08-01             0.0           0.0       -79600.10   
       1995-08-02             0.0           0.0       -79600.21   
       1995-08-03             0.0           0.0       -79600.10   
       1995-08-04             0.0           0.0       -79600.02   
...                           ...           ...             ...   
       2012-07-27             0.0           0.0       -90909.61   
       2012-07-28             0.0           0.0       -90876.52   
       2012-07-29             0.0           0.0       -90834.50   
       2012-07-30             0.0           0.0       -90790.57   
       2012-07-31             0.0           0.0       -90745.12   

                   Recharge_Maill  Évaporat_Maill  Excès_Irrigat  ...  \
zone   date                                                       ...   
global 1995-07-31    0.000000e+00             0.0            0.0  ...   
       1995-08-01    2.432245e-10             0.0            0.0  ...   
       1995-08-02    1.162146e-09             0.0            0.0  ...   
       1995-08-03    7.596600e-11             0.0            0.0  ...   
       1995-08-04    2.252325e-10             0.0            0.0  ...   
...                           ...             ...            ...  ...   
       2012-07-27    6.960876e+04             0.0            0.0  ...   
       2012-07-28    6.904120e+04             0.0            0.0  ...   
       2012-07-29    6.842738e+04             0.0            0.0  ...   
       2012-07-30    6.781939e+04             0.0            0.0  ...   
       2012-07-31    6.721684e+04             0.0            0.0  ...   

                   d_BilQ<->QFixé  Glob_Drain_Napp  Glob_Lacs_Nappe  \
zone   date                                                           
global 1995-07-31       -0.050260              0.0              0.0   
       1995-08-01       -0.013198              0.0              0.0   
       1995-08-02       -0.001269              0.0              0.0   
       1995-08-03       -0.011908              0.0              0.0   
       1995-08-04       -0.005316              0.0              0.0   
...                           ...              ...              ...   
       2012-07-27       -0.000052              0.0              0.0   
       2012-07-28       -0.000006              0.0              0.0   
       2012-07-29       -0.000062              0.0              0.0   
       2012-07-30       -0.000015              0.0              0.0   
       2012-07-31       -0.000016              0.0              0.0   

                   d_BilanVol_Pas  %_Non_CV_Gl_Pas  Nombre_Désatur  \
zone   date                                                          
global 1995-07-31       -0.050260     2.304057e-05             2.0   
       1995-08-01       -0.013194  